# 🏗️ E-Commerce Scraping Pipeline — Evrensel Şablon

> **NeuraNovaV Ekosistemi** | Trendyol pipeline'ından türetilmiş evrensel şablon  
> Yeni bir platform eklemek için sadece `[PLATFORM]` alanlarını doldurun.

---


## 📋 Şablon Kullanım Kılavuzu

Bu şablon Trendyol pipeline'ında kanıtlanmış mimarinin birebir kopyasıdır.  
Yeni bir platform (Hepsiburada, Amazon TR, Çiçeksepeti vb.) eklemek için:

1. `[PLATFORM]` → platform adı (örn: `hepsiburada`)
2. `[BASE_URL]` → hedef site ana URL'i (örn: `https://www.hepsiburada.com`)
3. `[PRODUCT_URL_PATTERN]` → ürün sayfası URL pattern'i (örn: `/p/...` veya `/pd/...`)
4. `[CATEGORY_URL_PATTERN]` → kategori sayfası URL pattern'i
5. `[PRICE_SELECTOR]` → fiyat için CSS/XPath selector
6. `[RATING_SELECTOR]` → puan için CSS/XPath selector
7. `[TITLE_SELECTOR]` → başlık için CSS/XPath selector
8. `[NEXT_PAGE_SELECTOR]` → sonraki sayfa linki selector'ı

---

## 📁 Proje Dosya Yapısı

```
[PLATFORM]_bot/
├── [PLATFORM]_bot/
│   ├── spiders/
│   │   ├── [PLATFORM].py          # Keşif örümceği (tüm metadata)
│   │   ├── fiyat_guncelle.py      # Hızlı fiyat güncelleyici
│   │   └── selector.py            # CSS/XPath tanımları (merkezi)
│   ├── pipelines.py               # MongoDB pipeline + telemetri
│   ├── middlewares.py             # User-Agent rotasyon middleware
│   ├── items.py                   # Scrapy item tanımları
│   └── settings.py                # Scrapy yapılandırması
├── dashboard.py                   # Streamlit izleme paneli
├── scrapy.cfg
└── README.md
```

---

## ⚙️ Adım 1 — `settings.py`

```python
# [PLATFORM]_bot/settings.py

BOT_NAME = "[PLATFORM]_bot"

SPIDER_MODULES = ["[PLATFORM]_bot.spiders"]
NEWSPIDER_MODULE = "[PLATFORM]_bot.spiders"

# ─── Anti-Ban Ayarları ────────────────────────────────────────────────────────
ROBOTSTXT_OBEY = False
COOKIES_ENABLED = True
DOWNLOAD_DELAY = 2          # Saniye cinsinden bekleme (gerekirse artır)
RANDOMIZE_DOWNLOAD_DELAY = True

AUTOTHROTTLE_ENABLED = True
AUTOTHROTTLE_START_DELAY = 1
AUTOTHROTTLE_MAX_DELAY = 10
AUTOTHROTTLE_TARGET_CONCURRENCY = 2.0

CONCURRENT_REQUESTS = 8
CONCURRENT_REQUESTS_PER_DOMAIN = 4

# ─── Middleware ────────────────────────────────────────────────────────────────
DOWNLOADER_MIDDLEWARES = {
    "[PLATFORM]_bot.middlewares.RotateUserAgentMiddleware": 400,
    # Proxy aktifleştirmek için:
    # "scrapy_rotating_proxies.middlewares.RotatingProxyMiddleware": 610,
    # "scrapy_rotating_proxies.middlewares.BanDetectionMiddleware": 620,
}

# ROTATING_PROXY_LIST_PATH = "proxies.txt"  # proxy dosyan varsa aç

# ─── Pipeline ─────────────────────────────────────────────────────────────────
ITEM_PIPELINES = {
    "[PLATFORM]_bot.pipelines.[PLATFORM_PASCAL]Pipeline": 300,
}

# ─── HTTP Cache (geliştirme sırasında kullanışlı) ─────────────────────────────
# HTTPCACHE_ENABLED = True
# HTTPCACHE_EXPIRATION_SECS = 86400

# ─── MongoDB Bağlantısı ───────────────────────────────────────────────────────
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB   = "neuranovav_[PLATFORM]"   # her platform için ayrı DB
```

---

## 📦 Adım 2 — `items.py`

```python
# [PLATFORM]_bot/items.py
import scrapy

class [PLATFORM_PASCAL]Item(scrapy.Item):
    # ─── Temel Bilgiler ───────────────────────────────────────────────────────
    url         = scrapy.Field()
    title       = scrapy.Field()
    category    = scrapy.Field()

    # ─── Fiyat & Puan ─────────────────────────────────────────────────────────
    price       = scrapy.Field()
    evaluation  = scrapy.Field()       # Puan (float)
    evaluation_len = scrapy.Field()    # Yorum sayısı (int)

    # ─── Zengin Metadata ──────────────────────────────────────────────────────
    attributes  = scrapy.Field()       # dict: {"Renk": "Siyah", "Beden": "M"}
    images      = scrapy.Field()       # list of str
    explanation = scrapy.Field()       # Ürün açıklaması

    # ─── Pipeline Kontrol Alanları ────────────────────────────────────────────
    # Bu alanlar spider'dan değil pipeline'dan doldurulur
    # scrape_mode: "full" | "price_only"
    scrape_mode = scrapy.Field()
```

---

## 🕸️ Adım 3 — `selector.py` (Merkezi Selector Tanımları)

```python
# [PLATFORM]_bot/spiders/selector.py
# ⚠️  BURASI EN ÇOK DEĞİŞTİRİLECEK KISIM — platform'a göre güncelleyin

class Selectors:
    # ─── Ürün Sayfası ─────────────────────────────────────────────────────────
    # Tarayıcı DevTools ile inspect yaparak bulun (F12 > Elements > Copy selector)

    TITLE        = "[TITLE_SELECTOR]"
    # Örnek Trendyol:  "h1.pr-new-br span"
    # Örnek Hepsiburada: "h1.product-name"

    PRICE        = "[PRICE_SELECTOR]"
    # Örnek Trendyol:  "span.prc-dsc"  veya  JSON-LD: script[type='application/ld+json']
    # Örnek Hepsiburada: "span[data-bind*='currentPrice']"

    RATING       = "[RATING_SELECTOR]"
    RATING_COUNT = "[RATING_COUNT_SELECTOR]"

    IMAGES       = "[IMAGES_SELECTOR]"
    DESCRIPTION  = "[DESCRIPTION_SELECTOR]"
    ATTRIBUTES   = "[ATTRIBUTES_SELECTOR]"    # tablo/liste CSS

    # ─── Kategori / Liste Sayfası ─────────────────────────────────────────────
    PRODUCT_LINK = "[PRODUCT_LINK_SELECTOR]"   # her ürün kartındaki 
    NEXT_PAGE    = "[NEXT_PAGE_SELECTOR]"       # "sonraki sayfa" butonu/linki

    # ─── URL Pattern ──────────────────────────────────────────────────────────
    BASE_URL         = "[BASE_URL]"
    PRODUCT_PATTERN  = "[PRODUCT_URL_PATTERN]"  # örn: r"/p/[A-Z0-9]+"
    CATEGORY_PATTERN = "[CATEGORY_URL_PATTERN]"

    # ─── Anti-Ban İpuçları ────────────────────────────────────────────────────
    # Bazı siteler fiyatı HTML'de değil JavaScript/JSON-LD içinde saklar.
    # Böyle durumlarda:
    #   import json, re
    #   raw = response.css("script[type='application/ld+json']::text").get()
    #   data = json.loads(raw)
    #   price = data.get("offers", {}).get("price")
```

---

## 🕷️ Adım 4 — Keşif Spider'ı `[PLATFORM].py`

```python
# [PLATFORM]_bot/spiders/[PLATFORM].py
import scrapy
import json
import re
from ..items import [PLATFORM_PASCAL]Item
from .selector import Selectors

class [PLATFORM_PASCAL]Spider(scrapy.Spider):
    name = "[PLATFORM]"
    allowed_domains = ["[BASE_DOMAIN]"]   # örn: "hepsiburada.com"

    # ─── Başlangıç URL'leri ───────────────────────────────────────────────────
    # Tarayıcıda kategori sayfalarını gezin, URL'leri buraya yapıştırın
    start_urls = [
        "[BASE_URL]/[KATEGORI_1]",
        "[BASE_URL]/[KATEGORI_2]",
        # İstediğiniz kadar ekleyin...
    ]

    custom_settings = {
        "CLOSESPIDER_ITEMCOUNT": 0,    # 0 = limitsiz; test için 100 yapın
    }

    # ─── Kategori Sayfası ─────────────────────────────────────────────────────
    def parse(self, response):
        # Ürün linklerini topla
        for href in response.css(Selectors.PRODUCT_LINK + "::attr(href)").getall():
            if re.search(Selectors.PRODUCT_PATTERN, href):
                yield response.follow(href, callback=self.parse_product)

        # Sonraki sayfaya geç
        next_page = response.css(Selectors.NEXT_PAGE + "::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)

    # ─── Ürün Sayfası ─────────────────────────────────────────────────────────
    def parse_product(self, response):
        item = [PLATFORM_PASCAL]Item()
        item["scrape_mode"] = "full"

        # URL temizleme — tracking parametrelerini at, kritikleri koru
        clean_url = self._clean_url(response.url)
        item["url"] = clean_url

        # ── Başlık ────────────────────────────────────────────────────────────
        item["title"] = response.css(Selectors.TITLE + "::text").get("").strip()

        # ── Fiyat ─────────────────────────────────────────────────────────────
        # YÖNTEM A: Doğrudan CSS selector
        raw_price = response.css(Selectors.PRICE + "::text").get("")

        # YÖNTEM B: JSON-LD (bazı siteler fiyatı buraya gömer)
        # raw = response.css("script[type='application/ld+json']::text").get("")
        # try:
        #     data = json.loads(raw)
        #     raw_price = str(data.get("offers", {}).get("price", ""))
        # except (json.JSONDecodeError, AttributeError):
        #     raw_price = ""

        item["price"] = self._parse_price(raw_price)

        # ── Puan ──────────────────────────────────────────────────────────────
        rating_text = response.css(Selectors.RATING + "::text").get("")
        item["evaluation"] = self._parse_rating(rating_text)

        count_text = response.css(Selectors.RATING_COUNT + "::text").get("0")
        item["evaluation_len"] = self._parse_int(count_text)

        # ── Kategori ──────────────────────────────────────────────────────────
        breadcrumb = response.css("[BREADCRUMB_SELECTOR] ::text").getall()
        item["category"] = " > ".join(b.strip() for b in breadcrumb if b.strip())

        # ── Görseller ─────────────────────────────────────────────────────────
        item["images"] = response.css(Selectors.IMAGES + "::attr(src)").getall()

        # ── Özellikler ────────────────────────────────────────────────────────
        # Her sitenin attribute tablosu farklıdır — geliştirirken inceleyin
        attributes = {}
        # Örnek: RenkSiyah
        # for row in response.css(Selectors.ATTRIBUTES):
        #     key = row.css(".key::text").get("").strip()
        #     val = row.css(".val::text").get("").strip()
        #     if key:
        #         attributes[key] = val
        item["attributes"] = attributes

        # ── Açıklama ──────────────────────────────────────────────────────────
        item["explanation"] = " ".join(
            response.css(Selectors.DESCRIPTION + " ::text").getall()
        ).strip()

        yield item

    # ─── Yardımcı Metodlar ────────────────────────────────────────────────────
    def _clean_url(self, url: str) -> str:
        """Tracking parametrelerini temizle, kritikleri koru."""
        from urllib.parse import urlparse, urlencode, parse_qs, urlunparse
        KEEP_PARAMS = {"boutiqueId", "merchantId"}   # platforma göre güncelleyin
        parsed = urlparse(url)
        params = {k: v for k, v in parse_qs(parsed.query).items() if k in KEEP_PARAMS}
        return urlunparse(parsed._replace(query=urlencode(params, doseq=True)))

    def _parse_price(self, raw: str) -> float | None:
        """'1.299,99 TL' → 1299.99"""
        clean = re.sub(r"[^\d,]", "", raw).replace(",", ".")
        # Binlik ayracı nokta ise: "1.299.99" → "1299.99"
        parts = clean.split(".")
        if len(parts) > 2:
            clean = "".join(parts[:-1]) + "." + parts[-1]
        try:
            return float(clean)
        except ValueError:
            return None

    def _parse_rating(self, raw: str) -> float | None:
        """'4,7' veya '4.7' → 4.7"""
        # Önceden compile edilmiş regex performansı artırır
        _RATING_RE = re.compile(r"(\d)[.,](\d)")
        m = _RATING_RE.search(raw)
        if m:
            return float(f"{m.group(1)}.{m.group(2)}")
        return None

    def _parse_int(self, raw: str) -> int:
        digits = re.sub(r"\D", "", raw)
        return int(digits) if digits else 0
```

---

## 🔄 Adım 5 — Hızlı Fiyat Güncelleyici `fiyat_guncelle.py`

```python
# [PLATFORM]_bot/spiders/fiyat_guncelle.py
import scrapy
import re
from pymongo import MongoClient
from ..items import [PLATFORM_PASCAL]Item
from .selector import Selectors
from scrapy.utils.project import get_project_settings

class FiyatGuncelleSpider(scrapy.Spider):
    """
    MongoDB'deki mevcut URL'leri okur ve sadece fiyat + puan günceller.
    Kategori traversal yok → çok daha hızlı.
    Günlük cronjob olarak çalıştırılmak üzere tasarlandı.
    """
    name = "fiyat_guncelle"
    allowed_domains = ["[BASE_DOMAIN]"]

    def start_requests(self):
        settings = get_project_settings()
        client = MongoClient(settings["MONGO_URI"])
        db = client[settings["MONGO_DB"]]

        # Tüm bilinen URL'leri çek
        urls = [doc["url"] for doc in db["products"].find({}, {"url": 1})]
        self.logger.info(f"Güncellenecek URL sayısı: {len(urls)}")
        client.close()

        for url in urls:
            yield scrapy.Request(url, callback=self.parse_price)

    def parse_price(self, response):
        item = [PLATFORM_PASCAL]Item()
        item["scrape_mode"] = "price_only"
        item["url"] = response.url

        # Sadece fiyat ve puan — ağır metadata çekilmiyor
        raw_price = response.css(Selectors.PRICE + "::text").get("")
        item["price"] = self._parse_price(raw_price)

        rating_text = response.css(Selectors.RATING + "::text").get("")
        item["evaluation"] = self._parse_rating(rating_text)

        count_text = response.css(Selectors.RATING_COUNT + "::text").get("0")
        item["evaluation_len"] = self._parse_int(count_text)

        yield item

    # Trendyol spider'dan kopyala — aynı yardımcı metodlar
    def _parse_price(self, raw):
        clean = re.sub(r"[^\d,]", "", raw).replace(",", ".")
        parts = clean.split(".")
        if len(parts) > 2:
            clean = "".join(parts[:-1]) + "." + parts[-1]
        try:
            return float(clean)
        except ValueError:
            return None

    def _parse_rating(self, raw):
        _RATING_RE = re.compile(r"(\d)[.,](\d)")
        m = _RATING_RE.search(raw)
        return float(f"{m.group(1)}.{m.group(2)}") if m else None

    def _parse_int(self, raw):
        digits = re.sub(r"\D", "", raw)
        return int(digits) if digits else 0
```

---

## 🛡️ Adım 6 — `pipelines.py` (5-Durum Telemetri)

```python
# [PLATFORM]_bot/pipelines.py
# Trendyol pipeline'ından birebir alınmıştır.
# Sadece collection adları platforma göre değişebilir.

from datetime import datetime, timezone
from pymongo import MongoClient, UpdateOne, ASCENDING
from pymongo.errors import BulkWriteError
from scrapy.utils.project import get_project_settings
import re

class [PLATFORM_PASCAL]Pipeline:

    HEARTBEAT_INTERVAL = 10   # Her N item'da bir MongoDB'ye heartbeat yaz

    def open_spider(self, spider):
        settings = get_project_settings()
        self.client = MongoClient(settings["MONGO_URI"])
        self.db = self.client[settings["MONGO_DB"]]

        # ─── Collection İndeksleri ────────────────────────────────────────────
        self.db["products"].create_index("url", unique=True)
        self.db["price_history"].create_index(
            [("url", ASCENDING), ("date", ASCENDING)], unique=True
        )

        # ─── İş Kaydı ─────────────────────────────────────────────────────────
        self.job_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
        self.stats = {
            "yeni_urun": 0,
            "yeni_gun_kaydi": 0,
            "gun_ici_degisim": 0,
            "drop_fiyatsiz": 0,
            "drop_hata": 0,
        }
        self.total_processed = 0
        self.db["jobs"].insert_one({
            "job_id": self.job_id,
            "platform": spider.name,
            "status": "Running",
            "start_time": datetime.now(timezone.utc).isoformat(),
            "stats": self.stats,
            "last_ping": datetime.now(timezone.utc).isoformat(),
        })

    def close_spider(self, spider):
        self.db["jobs"].update_one(
            {"job_id": self.job_id},
            {"$set": {
                "status": "Completed",
                "end_time": datetime.now(timezone.utc).isoformat(),
                "total_processed": self.total_processed,
                "stats": self.stats,
            }}
        )
        self.client.close()

    def process_item(self, item, spider):
        self.total_processed += 1
        today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        url = item.get("url")
        price = item.get("price")
        scrape_mode = item.get("scrape_mode", "full")

        # ─── Fiyat Kontrolü ───────────────────────────────────────────────────
        if price is None:
            self.stats["drop_fiyatsiz"] += 1
            self._heartbeat()
            return item

        if not isinstance(price, (int, float)) or price <= 0:
            self.stats["drop_hata"] += 1
            self._heartbeat()
            return item

        # ─── Ürün Metadata Upsert ─────────────────────────────────────────────
        if scrape_mode == "full":
            update_fields = {k: v for k, v in item.items()
                             if v is not None and k not in ("scrape_mode", "price",
                                                             "evaluation", "evaluation_len")}
            update_fields["last_seen"] = datetime.now(timezone.utc).isoformat()

            result = self.db["products"].update_one(
                {"url": url},
                {"$set": update_fields},
                upsert=True
            )
            is_new = result.upserted_id is not None
        else:
            is_new = False
            self.db["products"].update_one(
                {"url": url},
                {"$set": {"last_seen": datetime.now(timezone.utc).isoformat()}}
            )

        # ─── Fiyat Geçmişi ────────────────────────────────────────────────────
        existing = self.db["price_history"].find_one({"url": url, "date": today})

        if is_new:
            self.stats["yeni_urun"] += 1
            self.db["price_history"].insert_one({
                "url": url, "date": today,
                "price": price,
                "evaluation": item.get("evaluation"),
                "evaluation_len": item.get("evaluation_len"),
            })
        elif existing is None:
            self.stats["yeni_gun_kaydi"] += 1
            self.db["price_history"].insert_one({
                "url": url, "date": today,
                "price": price,
                "evaluation": item.get("evaluation"),
                "evaluation_len": item.get("evaluation_len"),
            })
        elif existing["price"] != price or existing.get("evaluation") != item.get("evaluation"):
            self.stats["gun_ici_degisim"] += 1
            self.db["price_history"].update_one(
                {"url": url, "date": today},
                {"$set": {
                    "price": price,
                    "evaluation": item.get("evaluation"),
                    "evaluation_len": item.get("evaluation_len"),
                }}
            )
        # else: aynı gün aynı fiyat → hiçbir şey yapma

        self._heartbeat()
        return item

    def _heartbeat(self):
        if self.total_processed % self.HEARTBEAT_INTERVAL == 0:
            self.db["jobs"].update_one(
                {"job_id": self.job_id},
                {"$set": {
                    "last_ping": datetime.now(timezone.utc).isoformat(),
                    "total_processed": self.total_processed,
                    "stats": self.stats,
                }}
            )
```

---


## 🔄 Adım 7 — `middlewares.py` (User-Agent Rotasyonu)

```python
# [PLATFORM]_bot/middlewares.py
import random

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 "
    "(KHTML, like Gecko) Version/17.3 Safari/605.1.15",
    "Mozilla/5.0 (X11; Linux x86_64; rv:124.0) Gecko/20100101 Firefox/124.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36 Edg/121.0.0.0",
    # Gerekirse daha fazla ekleyin — https://useragentstring.com/
]

class RotateUserAgentMiddleware:
    def process_request(self, request, spider):
        request.headers["User-Agent"] = random.choice(USER_AGENTS)
```

---

## 🗄️ Adım 8 — MongoDB Şeması

Trendyol pipeline'ı ile aynı şema. Sadece DB adı değişir.

### `products` — Ürün Kataloğu

```json
{
  "url": "[BASE_URL]/[PRODUCT_URL_PATTERN]/...",
  "title": "Ürün Başlığı",
  "category": "Ana Kategori > Alt Kategori > ...",
  "attributes": { "Renk": "Siyah", "Beden": "M" },
  "images": ["https://cdn.[PLATFORM].com/..."],
  "explanation": "Ürün açıklaması...",
  "last_seen": "2026-03-06T10:00:00Z"
}
```

> `url` alanı unique index — tüm ölçeklerde O(1) arama.

### `price_history` — Fiyat Zaman Serisi

```json
{
  "url": "[BASE_URL]/...",
  "date": "2026-03-06",
  "price": 1299.99,
  "evaluation": 4.7,
  "evaluation_len": 312
}
```

> `(url, date)` compound unique index — günlük duplicate önleme.

### `jobs` — Operasyonel Telemetri

```json
{
  "job_id": "20260306_100000",
  "platform": "[PLATFORM]",
  "status": "Completed",
  "start_time": "...",
  "end_time": "...",
  "total_processed": 5000,
  "stats": {
    "yeni_urun": 320,
    "yeni_gun_kaydi": 4200,
    "gun_ici_degisim": 480,
    "drop_fiyatsiz": 0,
    "drop_hata": 0
  },
  "last_ping": "..."
}
```

---

## 🐳 Adım 9 — Docker MongoDB

```bash
# Her platform için aynı MongoDB container'ı kullanılabilir — DB adı farklı olur.
docker run -d \
  --name neuranovav_mongo \
  --restart always \
  -p 27017:27017 \
  -v neuranovav_data:/data/db \
  mongo:7.0
```

---

# 🚀 Adım 10 — Çalıştırma

```bash
# 1. Sanal ortam
python -m venv venv
source venv/bin/activate       # Linux/Mac
# .\venv\Scripts\activate      # Windows

# 2. Bağımlılıklar
pip install scrapy pymongo pandas streamlit plotly streamlit-autorefresh scrapy-rotating-proxies

# 3. Dashboard başlat
streamlit run dashboard.py

# 4. Keşif spider (tam metadata)
scrapy crawl [PLATFORM]

# 5. Günlük fiyat güncelleme
scrapy crawl fiyat_guncelle
```

---

## 📊 Adım 11 — Dashboard `dashboard.py`

`dashboard.py` birebir Trendyol sürümüyle aynıdır.  
Sadece başındaki `MONGO_DB` değişkenini güncelleyin:

```python
# dashboard.py — değiştirilecek tek satır
MONGO_DB = "neuranovav_[PLATFORM]"   # örn: "neuranovav_hepsiburada"
```

---

## 🔍 Platform Entegrasyon Kontrol Listesi

Yeni bir platform eklerken sırayla yapılacaklar:

- [ ] `selector.py` → tüm `[...]` alanları gerçek selector ile doldur
- [ ] `settings.py` → `MONGO_DB` adını güncelle
- [ ] `[PLATFORM].py` → `start_urls` listesini doldur
- [ ] `_parse_price()` → platforma özgü fiyat formatını test et
- [ ] `_clean_url()` → `KEEP_PARAMS` setini platforma göre ayarla
- [ ] `dashboard.py` → `MONGO_DB` satırını güncelle
- [ ] Test: `scrapy crawl [PLATFORM] -s CLOSESPIDER_ITEMCOUNT=10`
- [ ] Test: MongoDB'de 10 ürünü doğrula
- [ ] Test: `scrapy crawl fiyat_guncelle -s CLOSESPIDER_ITEMCOUNT=5`
- [ ] Test: Dashboard'u aç, KPI'lar görünüyor mu?

---

# ⚠️ Platform-Spesifik Notlar

### Hepsiburada için ipuçları
- Ürün URL'leri `/p/HBVXXXXXXX` formatında
- Fiyat bazen JavaScript render gerektirebilir → `splash` veya `playwright` gerekebilir
- `merchantId` parametresi kritik — temizleme fonksiyonunda koru

### Amazon TR için ipuçları
- Anti-bot koruması çok güçlü → `DOWNLOAD_DELAY = 3-5` öneririz
- `ASIN` kodu URL'den çıkarılabilir → unique key olarak kullanılabilir
- Fiyat genellikle JSON-LD içinde (`PriceSpecification`)

### Genel İpuçları
- İlk denemede `HTTPCACHE_ENABLED = True` ile çalış — gereksiz istek yapmazsın
- Selector'ları bulmak için: DevTools → Elements → sağ tık → Copy → Copy selector
- Fiyat None dönüyorsa: önce JSON-LD'yi dene, sonra CSS

---

*NeuraNovaV Ekosistemi — Pipeline Şablonu v1.0 | Trendyol → Evrensel adaptasyon*
